# Sweetpotato Genomic Prediction

The study links 54 partially simulated phenotyped clones to 10,000 simulated genotype profiles with
500 SNP markers.

## 1. Methodology

### 1.1. Research Design, Data Sources and Analytical Framework

The unit of analysis is a clone. Raw phenotype records are partially curated
and partially synthetic; the SNP matrix and embedded marker-index signal are
simulated. The notebook reads the existing raw files; it does **not** regenerate
phenotypes or genotypes. Generation scripts are documented in
[the project guide](../docs/readme.md).

Select the project's Python environment with dependencies from
[requirements.txt](../docs/requirements.txt). 

In [1]:
# Import necessary libraries
# --- Setup: imports, project paths, and file locations ---
from __future__ import annotations
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from importlib import reload
from pathlib import Path
import subprocess
import sys

# h5py is installed automatically if it is missing, so the notebook runs on a fresh setup.
try:
    import h5py  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'h5py'])
    import h5py  # noqa: F401

import pandas as pd  # tables (DataFrames)
import plotly.express as px  # interactive charts


*Define paths, shared labels, plot styling, and the target used for candidate ranking.*

In [2]:
# Find the project folder whether the notebook is opened from notebooks/ or the project root.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError(f"Could not find project src directory under: {PROJECT_ROOT}")

# Let Python import our own code from the project's src/ folder.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# reload() picks up any edits made to the scripts without restarting the notebook.
import src.etl_pipeline as etl_pipeline
import src.capstone_pipeline as capstone_pipeline
reload(etl_pipeline)
reload(capstone_pipeline)
from src.etl_pipeline import run_etl  # cleans the raw CSV files
from src.capstone_pipeline import run_capstone_analysis  # trains models and ranks clones

# Input files: raw = messy originals, processed = cleaned copies created by the ETL step below.
RAW_PHENOTYPE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'raw_phenotypes.csv'
RAW_GENOTYPE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'raw_genotypes.csv'
PHENOTYPE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'phenotype_clean.csv'
GENOTYPE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'genotype_clean.csv'
# Results (the two summary CSV files) are saved here.
OUTPUT_DIR = PROJECT_ROOT / 'outputs'

# Print the paths so you can confirm the notebook found the right folders.
print(f'Project root:        {PROJECT_ROOT}')
print(f'Raw phenotype input: {RAW_PHENOTYPE_PATH}')
print(f'Raw genotype input:  {RAW_GENOTYPE_PATH}')
print(f'Output folder:       {OUTPUT_DIR}')


# --- Shared figure style so every plot looks consistent in the report ---
TARGET_NAMES = {
    'market_index': 'Market Index', 'dri': 'DRI', 'eqi': 'EQI',
    'parental_value': 'Parental Value', 'ayi': 'AYI',
}
RANKING_TARGET = 'market_index'

MODEL_NAMES = {'elastic_net': 'Elastic Net', 'ridge': 'Ridge', 'mlp_baseline': 'MLP'}
PALETTE = ['#0072B2', '#E69F00', '#009E73']  # colour-blind-safe
PLOT_COLORS = PALETTE

def style_figure(fig, width=600, height=400):
    """Publication style: no title (use the manuscript caption), large legible text, black axes."""
    fig.update_layout(
        template='plotly_white',
        title=None,
        font=dict(family='Arial', size=18, color='black'),
        legend=dict(font=dict(size=18), title=dict(font=dict(size=18))),
        width=width, height=height,
        margin=dict(l=100, r=40, t=40, b=100),
    )
    axis_style = dict(
        showline=True, linecolor='black', mirror=True, ticks='outside',
        tickfont=dict(size=16, color='black'), title_font=dict(size=20, color='black'),
    )
    fig.update_xaxes(**axis_style)
    fig.update_yaxes(**axis_style)
    return fig


ILLUSTRATIONS_DIR = PROJECT_ROOT / 'illustrations'
ILLUSTRATIONS_DIR.mkdir(exist_ok=True)

def save_figure(fig, name):
    """Save a figure as a 300-dpi-equivalent PNG (scale=3) in illustrations/."""
    fig.write_image(ILLUSTRATIONS_DIR / f'{name}.png', scale=3)


Project root:        C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\sweetpotato_capstone_notebook
Raw phenotype input: C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\sweetpotato_capstone_notebook\data\raw\raw_phenotypes.csv
Raw genotype input:  C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\sweetpotato_capstone_notebook\data\raw\raw_genotypes.csv
Output folder:       C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\sweetpotato_capstone_notebook\outputs


### 1.2. Data Pre-processing and Quality Checks

ETL normalizes clone identifiers, converts numeric and percent-formatted values, resolves duplicates, and calculates indices.
Missing SNP calls remain missing until median imputation inside each model's training fold. 


*Regenerate the processed CSVs and ETL audit report without modifying the raw inputs.*


In [3]:
# ETL = Extract, Transform, Load: read the messy raw files, fix them, and save clean copies.
etl_result = run_etl(
    raw_phenotype_path=RAW_PHENOTYPE_PATH,
    raw_genotype_path=RAW_GENOTYPE_PATH,
    clean_phenotype_path=PHENOTYPE_PATH,
    clean_genotype_path=GENOTYPE_PATH,
)

print('Data Pre-processing and Quality Checks:')
# Turn the ETL report (what was cleaned, dropped, or merged) into a readable table.
etl_report = etl_result['report']
etl_summary = pd.DataFrame(
    [
        {'stage': stage, 'metric': metric, 'value': value}
        for stage in ('phenotype', 'genotype')
        for metric, value in etl_report[stage].items()
    ]
    + [
        {
            'stage': 'alignment',
            'metric': 'overlapping_clone_ids_after_join',
            'value': etl_report['overlapping_clone_ids_after_join'],
        }
    ]
)
etl_summary


Data Pre-processing and Quality Checks:


,stage,metric,value
0,phenotype,rows_in,69
1,phenotype,exact_duplicate_rows_dropped,0
2,phenotype,duplicate_clone_id_rows_resolved,15
3,phenotype,rows_dropped_missing_target,0
4,phenotype,final_row_count,54
5,genotype,rows_in,10005
6,genotype,malformed_marker_cells_coerced_to_nan,0
7,genotype,exact_duplicate_rows_dropped,5
8,genotype,duplicate_clone_id_rows_merged,0
9,genotype,missing_marker_calls,15134


*Phenotype and genotype datasets structure*

In [4]:
# Display the structure of the cleaned phenotype and genotype datasets
data_structure = pd.DataFrame([
    {
        'Dataset': 'Phenotypes',
        'Raw rows': etl_report['phenotype']['rows_in'],
        'Clean rows': len(etl_result['phenotype']),
        'Clean columns': etl_result['phenotype'].shape[1],
    },
    {
        'Dataset': 'Genotypes',
        'Raw rows': etl_report['genotype']['rows_in'],
        'Clean rows': len(etl_result['genotype']),
        'Clean columns': etl_result['genotype'].shape[1],
    },
])
print('Table 1 companion: current input dimensions (columns include clone_id)')
display(data_structure)
print(f"Overlapping clone IDs: {etl_report['overlapping_clone_ids_after_join']}")

Table 1 companion: current input dimensions (columns include clone_id)


,Dataset,Raw rows,Clean rows,Clean columns
0,Phenotypes,69,54,27
1,Genotypes,10005,10000,501


Overlapping clone IDs: 54


*Display raw and processed datasets*

In [5]:
# Display raw and processed phenotype data for comparison
print('Raw phenotypes: first 5 rows (original text and missing-value tokens)')
display(pd.read_csv(RAW_PHENOTYPE_PATH, nrows=5, dtype=str, keep_default_na=False).head())

print('Processed phenotypes: first 5 rows')
pheno = pd.read_csv(PHENOTYPE_PATH)
display(pheno.head())

Raw phenotypes: first 5 rows (original text and missing-value tokens)


,clone_id,variety,origin,release_year,skin_color,flesh_color,maturity_dap_min,maturity_dap_max,dry_matter_pct_literature,fusarium_wilt_resistance,...,southern_rkn_resistance,market_category,yield_kg_measured,pct_dm_measured,pct_starch_measured,beta_carotene_measured,fwilt_measured,ssr_measured,srkn_measured,grkn_measured
0,AVERRE,Averre,Virginia,1996,red,yellow-to-salmon orange,110,130,32.4,0.764,...,0.257,Dual-purpose,1.4401,31.1835,15.6201,0.3805,0.729,0.272,0.326,0.143
1,BAYOU BELLE,Bayou Belle,LSU AgCenter,2012,red-purple,deep orange,90,105,23.3,0.792,...,0.838,French fry processing,0.9844,22.5238,11.2619,0.5058,0.756,0.919,0.871,0.973
2,BAYOU BELLE,Bayou Belle,LSU AgCenter,2012,red-purple,deep orange,90,100,22.6,0.5,...,0.85,French fry processing / high yield / short cycle,0.9685,22.1598,11.0799,0.4976,0.85,1,0.7,0.7
3,BEAUREGARD,Beauregard,LSU AgCenter,1987,rose/light copper,orange,100,120,29.7,0.765,...,0.81,Fresh market benchmark,0.8769,29.8009,15.8973%,0.2754,0.581,0.564,0.86,0.82
4,BEAUREGARD P1-E12,Beauregard,LSU AgCenter,1987,rose/light copper,orange,110,120,27.8,1,...,0,International benchmark fresh market,0.9564,28.0878,14.0439,0.386,0.65,0.625,0.075,0


Processed phenotypes: first 5 rows


,clone_id,variety,origin,release_year,skin_color,flesh_color,maturity_dap_min,maturity_dap_max,dry_matter_pct_literature,fusarium_wilt_resistance,...,fwilt_measured,ssr_measured,srkn_measured,grkn_measured,n_measured_sources,dri,eqi,ayi,market_index,parental_value
0,AVERRE,Averre,Virginia,1996,red,yellow-to-salmon orange,110,130,32.4,0.764,...,0.729,0.272,0.326,0.143,8,0.403714,0.570072,0.588367,0.520718,0.510847
1,BAYOU BELLE,Bayou Belle,LSU AgCenter,2012,red-purple,deep orange,90,105,23.3,0.792,...,0.756,0.919,0.871,0.973,8,0.850143,0.783381,0.728133,0.787219,0.787987
2,BEAUREGARD,Beauregard,LSU AgCenter,1987,rose/light copper,orange,100,120,29.7,0.765,...,0.581,0.564,0.860,0.820,8,0.689571,0.556248,0.608967,0.618262,0.630665
3,BEAUREGARD P1-E12,Beauregard,LSU AgCenter,1987,rose/light copper,orange,110,120,27.8,1.000,...,0.650,0.625,0.075,0.000,8,0.457143,0.624092,0.552133,0.544456,0.528529
4,BEAUREGARD P2-E10,Beauregard,LSU AgCenter,1987,rose/light copper,orange,110,120,25.9,1.000,...,0.650,0.625,0.075,0.000,8,0.457143,0.678300,0.521033,0.552159,0.526930


In [6]:
# Display raw and processed genotype data for comparison
print('Raw genotypes: first 5 rows (original text and missing-value tokens)')
display(pd.read_csv(RAW_GENOTYPE_PATH, nrows=5, dtype=str, keep_default_na=False).head())

print('Processed genotypes: first 5 rows')
display(pd.read_csv(GENOTYPE_PATH, nrows=5).head())

Raw genotypes: first 5 rows (original text and missing-value tokens)


,clone_id,SNP_0001,SNP_0002,SNP_0003,SNP_0004,SNP_0005,SNP_0006,SNP_0007,SNP_0008,SNP_0009,...,SNP_0491,SNP_0492,SNP_0493,SNP_0494,SNP_0495,SNP_0496,SNP_0497,SNP_0498,SNP_0499,SNP_0500
0,AVERRE,1.0,2.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,...,0.0,0.0,0.0,0.0,2.0,0.0,2.0,1.0,2.0,1.0
1,BAYOU BELLE,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,2.0,...,0.0,2.0,1.0,0.0,2.0,0.0,2.0,2.0,0.0,0.0
2,BEAUREGARD,1.0,1.0,0.0,1.0,1.0,0.0,1.0,2.0,2.0,...,2.0,2.0,0.0,2.0,2.0,1.0,1.0,2.0,2.0,2.0
3,BEAUREGARD P1-E12,1.0,2.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,2.0,2.0,1.0,2.0,1.0,1.0,2.0,2.0,2.0
4,BEAUREGARD P2-E10,1.0,2.0,0.0,1.0,0.0,1.0,1.0,1.0,2.0,...,0.0,0.0,1.0,0.0,2.0,0.0,0.0,1.0,2.0,0.0


Processed genotypes: first 5 rows


,clone_id,SNP_0001,SNP_0002,SNP_0003,SNP_0004,SNP_0005,SNP_0006,SNP_0007,SNP_0008,SNP_0009,...,SNP_0491,SNP_0492,SNP_0493,SNP_0494,SNP_0495,SNP_0496,SNP_0497,SNP_0498,SNP_0499,SNP_0500
0,AVERRE,1.0,2.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,...,0.0,0.0,0.0,0.0,2.0,0.0,2.0,1.0,2.0,1.0
1,BAYOU BELLE,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,2.0,...,0.0,2.0,1.0,0.0,2.0,0.0,2.0,2.0,0.0,0.0
2,BEAUREGARD,1.0,1.0,0.0,1.0,1.0,0.0,1.0,2.0,2.0,...,2.0,2.0,0.0,2.0,2.0,1.0,1.0,2.0,2.0,2.0
3,BEAUREGARD P1-E12,1.0,2.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,2.0,2.0,1.0,2.0,1.0,1.0,2.0,2.0,2.0
4,BEAUREGARD P2-E10,1.0,2.0,0.0,1.0,0.0,1.0,1.0,1.0,2.0,...,0.0,0.0,1.0,0.0,2.0,0.0,0.0,1.0,2.0,0.0


### 1.3. Trait Scoring

ETL calls [scoring_pipeline.py](../src/scoring_pipeline.py), which calculates:

| Target column | Interpretation |
|---|---|
| `dri` | Disease Resistance Index: mean of available normalized resistance measurements |
| `eqi` | Eating Quality Index: equal-weight sweetness, flesh-color, and moistness proxies |
| `ayi` | Agronomic Yield Index: available normalized yield and maturity-preference components |
| `market_index` | Market Index: `(DRI + EQI + AYI) / 3` |
| `parental_value` | Parental Value: `0.4 * DRI + 0.4 * AYI + 0.2 * EQI` |

All five scores use a 0-1 scale. Descriptor-based fallback rules and subjective
weights make these proxy indices, not direct measurements of breeding value.


*Target columns*

In [7]:
# Define the target columns for genomic prediction analysis
TARGET_COLS = [
    'market_index',
    'dri',
    'eqi',
    'parental_value',
    'ayi',
]


### 1.4. Predictive Models and Cross-Validation

The shared engine is [capstone_pipeline.py](../src/capstone_pipeline.py).
Each target has a separate regression model:

- **Elastic Net (primary):** median imputation, univariate marker selection,
  scaling, and `ElasticNetCV` with L1/L2 regularization.
- **Ridge:** median imputation, scaling, and `RidgeCV` using all markers.
- **MLP:** median imputation, marker selection, scaling, PCA (at most 8
  components), and one 8-unit ReLU hidden layer with L2 regularization and
  `lbfgs` optimization.

The results below use shuffled 5-fold out-of-fold (OOF) evaluation with the shared seed. 
Outer held-out clones are excluded from preprocessing and model fitting. 
R-squared measure explained variation (higher is better); 
RMSE and MAE measure error in index units (lower is better). 

## 2. Results
### 2.1. Data Structure


*Distribution of target columns*

In [8]:
# Summarize the distribution of target columns
distribution_rows = []
for column in TARGET_COLS:
    values = pheno[column].dropna()
    q1, median, q3 = values.quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    distribution_rows.append({
        'Target': TARGET_NAMES[column],
        'n': len(values),
        'Median': median,
        'IQR': iqr,
        'Skewness': stats.skew(values, bias=False),
        'Shapiro-Wilk p': stats.shapiro(values).pvalue,
        '1.5-IQR outliers': ((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum(),
    })
distribution_summary = pd.DataFrame(distribution_rows)
distribution_summary.round(4)

,Target,n,Median,IQR,Skewness,Shapiro-Wilk p,1.5-IQR outliers
0,Market Index,54,0.5493,0.1071,0.0341,0.9423,2
1,DRI,54,0.5214,0.2262,0.2109,0.6412,0
2,EQI,54,0.6447,0.1043,-0.5967,0.1810,2
3,Parental Value,54,0.5287,0.1346,0.1321,0.8604,0
4,AYI,54,0.5125,0.2331,-0.1451,0.0710,0


*Target distribution plots" Histogram*

In [9]:
# Visualize the distribution of target columns
long_targets = (
    pheno[TARGET_COLS]
    .rename(columns=TARGET_NAMES)
    .melt(var_name='target', value_name='value')
)

fig = px.histogram(
    long_targets,
    x='value',
    facet_col='target',
    facet_col_wrap=3,
    nbins=12,
    category_orders={
        'target': [TARGET_NAMES[col] for col in TARGET_COLS]
    },
    color_discrete_sequence=[PLOT_COLORS[0]],
    labels={'value': '', 'count': ''},
    facet_col_spacing=0.08,
    facet_row_spacing=0.12,
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_traces(marker_line_color='white', marker_line_width=1)

fig.update_xaxes(title_text='', showticklabels=False, ticks='', showgrid=True, gridcolor='#E6E6E6', zeroline=False)
fig.update_yaxes(title_text='', showgrid=True, gridcolor='#E6E6E6', zeroline=False)

fig.add_annotation(
    text='Index value (0 - 1 scale)',
    xref='paper',
    yref='paper',
    x=0.5,
    y=-0.12,
    showarrow=False,
)
fig.add_annotation(
    text='Number of clones',
    xref='paper',
    yref='paper',
    x=-0.095,
    y=0.5,
    textangle=-90,
    showarrow=False,
)

fig.update_layout(
    template='plotly_white',
    width=800,
    height=600,
    showlegend=False,
    margin=dict(l=85, r=25, t=55, b=65),
    font=dict(size=13),
)
fig.update_annotations(font_size=15)

save_figure(fig, 'fig01_target_distributions')
fig.show(config={'responsive': True})

*Target QQ plots to assess normality* 

In [10]:
# Create QQ plots for each target column to assess normality.
q_fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[TARGET_NAMES[col] for col in TARGET_COLS],
    horizontal_spacing=0.12,
    vertical_spacing=0.12,
)

for index, col in enumerate(TARGET_COLS):
    row, column = divmod(index, 2)
    row += 1
    column += 1

    values = pheno[col].dropna().to_numpy()
    (theoretical, observed), (slope, intercept, _) = stats.probplot(
        values,
        dist='norm',
    )

    line_x = [theoretical[0], theoretical[-1]]
    line_y = [slope * x + intercept for x in line_x]

    q_fig.add_trace(
        go.Scatter(
            x=theoretical,
            y=observed,
            mode='markers',
            marker=dict(color=PLOT_COLORS[0], size=7),
            showlegend=False,
        ),
        row=row,
        col=column,
    )
    q_fig.add_trace(
        go.Scatter(
            x=line_x,
            y=line_y,
            mode='lines',
            line=dict(color=PLOT_COLORS[1], dash='dash', width=2),
            showlegend=False,
        ),
        row=row,
        col=column,
    )       

# Hide the unused sixth panel.
q_fig.update_xaxes(visible=False, row=3, col=2)
q_fig.update_yaxes(visible=False, row=3, col=2)

q_fig.update_xaxes(title_text='', showgrid=True, gridcolor='#E6E6E6', zeroline=False)
q_fig.update_yaxes(title_text='', showgrid=True, gridcolor='#E6E6E6', zeroline=False)

q_fig.add_annotation(
    text='Theoretical normal quantiles',
    xref='paper',
    yref='paper',
    x=0.5,
    y=-0.12,
    xanchor='center',
    yanchor='top',
    showarrow=False,
)
q_fig.add_annotation(
    text='Observed target values',
    xref='paper',
    yref='paper',
    x=-0.095,
    y=0.5,
    xanchor='center',
    yanchor='middle',
    textangle=-90,
    showarrow=False,
)

q_fig.update_layout(
    template='plotly_white',
    width=800,
    height=600,
    margin=dict(l=100, r=25, t=60, b=95),
    font=dict(size=12),
)
q_fig.update_annotations(font_size=14)

save_figure(q_fig, 'fig02_target_qqplots')
q_fig.show(config={'responsive': True})

*Box plots showing index IQR and outliers*

In [11]:
# Box plots show the interquartile range (box) and 1.5 x IQR outliers for each index.
box_targets = pheno[TARGET_COLS].rename(columns=TARGET_NAMES).melt(
    var_name='target', value_name='value'
)

iqr_fig = px.box(
    box_targets,
    x='target',
    y='value',
    points='outliers',
    category_orders={'target': [TARGET_NAMES[col] for col in TARGET_COLS]},
    color_discrete_sequence=[PALETTE[0]],
    labels={'target': 'Index', 'value': 'Standardized Index values'},
)
iqr_fig.update_traces(
    marker=dict(color=PALETTE[1], size=8),
    line=dict(color=PALETTE[0]),
    fillcolor='rgba(0, 114, 178, 0.25)',
)
style_figure(iqr_fig, width=600, height=400)
iqr_fig.update_xaxes(title_text='Target index')
iqr_fig.update_yaxes(title_text='Index value (0 - 1 scale)')
save_figure(iqr_fig, 'fig06_target_iqr_outliers')
iqr_fig.show(config={'responsive': True})

### 2.2. Elastic Net Prediction Power


*Run the prediction pipeline and check clones that had both genotype and phenotype data (training set)*

In [12]:
# Run the capstone analysis pipeline and evaluate input data alignment and cross-validated performance
# Run the capstone analysis pipeline for the specified targets
results = run_capstone_analysis( 
    phenotype_path=PHENOTYPE_PATH,
    genotype_path=GENOTYPE_PATH,
    output_dir=OUTPUT_DIR,
    targets=TARGET_COLS,
)

# How many clones had both genotype and phenotype data (the training set)?
print('Input data alignment')
display(results['alignment'])


Input data alignment


,metric,value
0,phenotype clones,54
1,genotype clones,10000
2,overlapping clones,54
3,clones with observed target,54
4,SNP markers,500


*Elastic Net cross-validated performance metrics*

In [13]:
# Display cross-validated performance metrics for each target
print('Performance calculated from pooled out-of-fold predictions')
metrics_table = pd.DataFrame(results['metrics']).T
performance_table = (
    metrics_table[
        ['model_family', 'n_labeled_samples', 'n_cv_folds', 'cv_r2', 'cv_rmse', 'cv_mae']
    ]
    .rename(index=TARGET_NAMES, columns={
        'model_family': 'Model',
        'n_labeled_samples': 'Labeled clones',
        'n_cv_folds': 'CV folds',
        'cv_r2': 'OOF R-squared',
        'cv_rmse': 'RMSE',
        'cv_mae': 'MAE',
    })
    .rename_axis('Target index')
)
performance_table['Model'] = performance_table['Model'].map(MODEL_NAMES)
performance_table.style.format({
    'Labeled clones': '{:.0f}', 'CV folds': '{:.0f}',
    'OOF R-squared': '{:.3f}', 'RMSE': '{:.3f}', 'MAE': '{:.3f}',
}, na_rep='Not evaluated').set_caption(
    'Primary-model performance: higher R-squared and lower errors are better'
    )

Performance calculated from pooled out-of-fold predictions


,Model,Labeled clones,CV folds,OOF R-squared,RMSE,MAE
Target index,,,,,,
Market Index,Elastic Net,54,5,0.884,0.029,0.020
DRI,Elastic Net,54,5,0.767,0.074,0.054
EQI,Elastic Net,54,5,0.861,0.034,0.026
Parental Value,Elastic Net,54,5,0.905,0.029,0.022
AYI,Elastic Net,54,5,0.873,0.050,0.042


*Plotting observed vs. predicted (cross-validated) values to check prediction accuracy*


In [14]:
# Scatter plot of observed vs cross-validated predicted values for the diagnostic target
DIAGNOSTIC_TARGET = 'market_index'
diagnostic_rankings = results['targets'][DIAGNOSTIC_TARGET]['ranked_clones']
training_clones = diagnostic_rankings[diagnostic_rankings['is_training_clone']]

# x = measured value, y = prediction made without seeing that clone (cross-validated).
from sklearn.metrics import r2_score
target_label = TARGET_NAMES.get(DIAGNOSTIC_TARGET, DIAGNOSTIC_TARGET)
fig = px.scatter(
    training_clones,
    x=f'observed_{DIAGNOSTIC_TARGET}',
    y=f'cv_predicted_{DIAGNOSTIC_TARGET}',
    hover_name='clone_id',
    color_discrete_sequence=[PALETTE[0]],
)
fig.update_traces(marker=dict(size=10, opacity=0.8, line=dict(color='white', width=1)))

minimum = min(
    training_clones[f'observed_{DIAGNOSTIC_TARGET}'].min(),
    training_clones[f'cv_predicted_{DIAGNOSTIC_TARGET}'].min(),
)
maximum = max(
    training_clones[f'observed_{DIAGNOSTIC_TARGET}'].max(),
    training_clones[f'cv_predicted_{DIAGNOSTIC_TARGET}'].max(),
)
# Dashed y = x line: points on it are perfect predictions; the closer, the better.
fig.add_shape(type='line', x0=minimum, y0=minimum, x1=maximum, y1=maximum, line=dict(color='gray', dash='dash'))
cv_r2 = r2_score(training_clones[f'observed_{DIAGNOSTIC_TARGET}'], training_clones[f'cv_predicted_{DIAGNOSTIC_TARGET}'])
fig.add_annotation(text=f'CV R\u00b2 = {cv_r2:.2f}<br>n = {len(training_clones)} clones', xref='paper', yref='paper',
                   x=0.03, y=0.97, showarrow=False, align='left', xanchor='left', yanchor='top',
                   bgcolor='white', bordercolor='black', borderwidth=1, font=dict(size=18))
fig.update_xaxes(title_text=f'Observed {target_label} (0-1 scale)')
fig.update_yaxes(title_text=f'Predicted (cross-validated) {target_label} (0-1 scale)', scaleanchor='x', scaleratio=1)
style_figure(fig, width=600, height=600)
save_figure(fig, 'fig05_observed_vs_predicted')
fig.show(config={'responsive': True})


### 2.3. Model Comparison: Elastic Net, Ridge, Multilayer Perceptron (MLP)

#### Prediction Accuracy 


*Compare model families across targets using the same cross-validation splits*

In [15]:
# Import and reload the capstone pipeline module to pick up any changes
import importlib
import src.capstone_pipeline as capstone_pipeline

capstone_pipeline = importlib.reload(capstone_pipeline) # Reload the module to pick up any changes
compare_model_families_across_targets = capstone_pipeline.compare_model_families_across_targets # Function to compare model families across targets
TARGET_COLUMNS = capstone_pipeline.TARGET_COLUMNS 

# Same cross-validation splits for every model family, so the scores are a fair comparison.
MODEL_FAMILIES_TO_COMPARE = ("ridge", "elastic_net", "mlp_baseline")

print('Comparing model families across targets...')
model_comparison = compare_model_families_across_targets(
    phenotype_path=PHENOTYPE_PATH,
    genotype_path=GENOTYPE_PATH,
    output_dir=OUTPUT_DIR,
    model_families=MODEL_FAMILIES_TO_COMPARE,
)
table = (
    model_comparison.assign(
        Target=model_comparison['target'].map(TARGET_NAMES),
        Model=model_comparison['model_family'].map(MODEL_NAMES),
    )
    .sort_values(['Target', 'cv_r2'], ascending=[True, False])
    [['Target', 'Model', 'cv_r2', 'cv_rmse', 'cv_mae']]
    .rename(columns={'cv_r2': 'R-squared', 'cv_rmse': 'RMSE', 'cv_mae': 'MAE'})
)

table.round(3)

Comparing model families across targets...


,Target,Model,R-squared,RMSE,MAE
12,AYI,MLP,0.892,0.046,0.037
13,AYI,Elastic Net,0.873,0.050,0.042
14,AYI,Ridge,0.748,0.071,0.055
3,DRI,Ridge,0.820,0.065,0.051
4,DRI,MLP,0.791,0.070,0.053
5,DRI,Elastic Net,0.767,0.074,0.054
6,EQI,Elastic Net,0.861,0.034,0.026
7,EQI,MLP,0.844,0.036,0.025
8,EQI,Ridge,0.643,0.054,0.039
0,Market Index,Elastic Net,0.884,0.029,0.020


*Bar plot of the different models performance for each trait*

In [16]:
# Bars compare model families on each target; taller = better predictions.
plot_df = model_comparison.assign(
    Model=model_comparison['model_family'].map(MODEL_NAMES),
    Target=model_comparison['target'].map(TARGET_NAMES),
)
fig = px.bar(
    plot_df,
    x='Target',
    y='cv_r2',
    color='Model',
    barmode='group',
    category_orders={
        'Target': [TARGET_NAMES[t] for t in TARGET_COLUMNS],
        'Model': ['Elastic Net', 'Ridge', 'MLP'],
    },
    color_discrete_sequence=PALETTE,
)
style_figure(fig, width=600, height=400)
fig.update_yaxes(title_text='Cross-validated R\u00b2 (out-of-fold)', range=[0, 1.1], dtick=0.2, tickfont=dict(size=17), title_font=dict(size=21))
fig.update_xaxes(title_text='Target index', tickfont=dict(size=17), title_font=dict(size=21))
fig.update_layout(font=dict(size=19), legend=dict(title=dict(text='Model', font=dict(size=19)), font=dict(size=19), orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1), margin=dict(l=100, r=40, t=70, b=100))
save_figure(fig, 'fig03_model_comparison')
fig.show(config={'responsive': True})


### 2.4. Model Robustness and Stability

#### Computational Performance and Save/Reload Checks


*Measure Market Index CV, final fitting time, batch prediction time, serialized pipeline size, and save/reload equivalence*

In [17]:
import importlib
import io
import time
import joblib
import numpy as np
import pandas as pd
import src.capstone_pipeline as cp

cp = importlib.reload(cp) # Reload the capstone_pipeline module to ensure the latest changes are used

phenotype, genotype = cp.load_inputs(PHENOTYPE_PATH, GENOTYPE_PATH)
cp.validate_inputs(phenotype, genotype) # Validate that the input phenotype and genotype data are consistent and correctly formatted

target = "market_index"
target_label = TARGET_NAMES.get(target, target)
n_repeats = 5
seed = cp.DEFAULT_SEED

markers = genotype.columns.drop("clone_id").tolist()
labeled = phenotype.merge(genotype, on="clone_id", how="inner")
prediction_batch = genotype[markers]
benchmark_rows = []

# Repeat model fitting and timing. Fixed seeds mean the CV scores are not
# independent across repeats; use these repeats to summarize timing only.
for family in cp.MODEL_FAMILIES:
    for run in range(n_repeats):
        start = time.perf_counter()
        outcome = cp._fit_and_evaluate_target(
            target,
            labeled[markers],
            labeled[cp.TARGET_SOURCE_COLUMNS[target]],
            model_family=family,
            seed=seed,
        )
        fit_seconds = time.perf_counter() - start # Measure the time taken to fit the model

        if outcome.get("skipped"):
            raise RuntimeError(f"{family}: {outcome['skip_reason']}")

        model = outcome["_fitted_model"]

        # Warm up, then measure five predictions of the full 10,000-clone batch.
        reference = model.predict(prediction_batch)
        prediction_times = []
        for _ in range(5):
            start = time.perf_counter()
            model.predict(prediction_batch)
            prediction_times.append(time.perf_counter() - start) # Measure the time taken for each prediction run

        # Measure serialized size and verify that save/reload preserves predictions.
        with io.BytesIO() as buffer:
            joblib.dump(model, buffer, compress=0) 
            model_kib = buffer.tell() / 1024
            buffer.seek(0)
            restored = joblib.load(buffer) 

        np.testing.assert_allclose(
            reference,
            restored.predict(prediction_batch), # Verify that the restored model produces the same predictions as the original model
            rtol=1e-10,
            atol=1e-12,
        )

        benchmark_rows.append({
            "target": target_label,
            "model": family,
            "run": run + 1,
            "CV folds": outcome["n_cv_folds"],
            "CV R²": outcome["cv_r2"],
            "CV + final-fit seconds": fit_seconds,
            "median prediction ms": np.median(prediction_times) * 1000,
            "pipeline size KiB": model_kib,
            "save/reload check": "Passed"
            })

runs = pd.DataFrame(benchmark_rows)

# Summarize runtime-related measurements across repeats.
summary = runs.groupby(["target", "model"], as_index=False).agg(
    **{
        "CV folds": ("CV folds", "first"),
        "CV R²": ("CV R²", "first"),
        "fit time mean (s)": ("CV + final-fit seconds", "mean"),
        "fit time SD (s)": ("CV + final-fit seconds", "std"),
        "prediction mean (ms)": ("median prediction ms", "mean"),
        "prediction SD (ms)": ("median prediction ms", "std"),
        "pipeline size mean (KiB)": ("pipeline size KiB", "mean"),
        "save/reload check": ("save/reload check", "first")
        
        
    }
)

print(
    f"Computational benchmark\n"
    f"Target: {target_label}\n"
    f"Repeats per configuration: {n_repeats}\n"
    f"Current configuration: {outcome['n_cv_folds']} CV folds, seed {seed}\n"
    f"CV folds are selected by sample size (maximum {cp.MAX_CV_FOLDS})"
)
display(summary.round(3))

Computational benchmark
Target: Market Index
Repeats per configuration: 5
Current configuration: 5 CV folds, seed 42
CV folds are selected by sample size (maximum 5)


,target,model,CV folds,CV R²,fit time mean (s),fit time SD (s),prediction mean (ms),prediction SD (ms),pipeline size mean (KiB),save/reload check
0,Market Index,elastic_net,5,0.884,0.593,0.066,52.430,1.482,24.301,Passed
1,Market Index,mlp_baseline,5,0.876,0.260,0.060,53.033,1.611,28.387,Passed
2,Market Index,ridge,5,0.880,0.164,0.059,69.230,0.639,26.678,Passed


### 2.5. Model explainability

Ridge spreads effects over all SNPs; Elastic Net selects a smaller, regularized
subset; MLP predictions act through PCA components and nonlinear hidden units.
Correlated markers can redistribute linear coefficients.


### 2.6. Indexed Traits Associations

*Spearman correlations of the scored phenotypes. Stars use Benjamini–Hochberg-adjusted p-values across all unique index pairs.*

In [18]:
# Compute and visualize Spearman correlations between target traits
n_targets = len(TARGET_COLS)
rho = np.full((n_targets, n_targets), np.nan)
star_labels = np.full((n_targets, n_targets), '', dtype=object)
pair_indices = []
pair_p_values = []
adjusted_p = np.full((n_targets, n_targets), np.nan)

for i in range(n_targets):
    rho[i, i] = 1.0

    for j in range(i):
        pair = pheno[[TARGET_COLS[i], TARGET_COLS[j]]].dropna()
        r, p_value = stats.spearmanr(
            pair[TARGET_COLS[i]],
            pair[TARGET_COLS[j]],
        )

        rho[i, j] = r
        if not np.isfinite(p_value):
            raise ValueError(f'Undefined Spearman test for {TARGET_COLS[i]} and {TARGET_COLS[j]}.')
        pair_indices.append((i, j))
        pair_p_values.append(p_value)

# Adjust each unique pair once; exclude the diagonal and mirrored pairs.
p_values = np.asarray(pair_p_values)
order = np.argsort(p_values)
ranked_p = p_values[order] * len(p_values) / np.arange(1, len(p_values) + 1)
bh_p_values = np.empty_like(p_values)
bh_p_values[order] = np.minimum(1.0, np.minimum.accumulate(ranked_p[::-1])[::-1])

for (i, j), p_adjusted in zip(pair_indices, bh_p_values):
    adjusted_p[i, j] = p_adjusted
    star_labels[i, j] = (
            '***' if p_adjusted < 0.001 else
            '**' if p_adjusted < 0.01 else
            '*' if p_adjusted < 0.05 else
            ''
        )

fig = go.Figure(
    go.Heatmap(
        z=rho,
        x=[TARGET_NAMES[col] for col in TARGET_COLS],
        y=[TARGET_NAMES[col] for col in TARGET_COLS],
        zmin=-1,
        zmax=1,
        colorscale='RdBu_r',
        text=star_labels,
        customdata=adjusted_p,
        hovertemplate='%{y} vs %{x}<br>Spearman ρ: %{z:.3f}<br>BH-adjusted p: %{customdata:.3g}<extra></extra>',
        texttemplate='%{text}',
        textfont=dict(size=18),
        colorbar=dict(
            title=dict(text='Spearman ρ'),
            thickness=15,
        ),
        hoverongaps=False,
        connectgaps=False,
    )
)

fig.update_layout(
    template='plotly_white',
    title=dict(
        text=(
            'Spearman correlations'
            '<br><sup>* adjusted p &lt; 0.05, ** &lt; 0.01, '
            '*** &lt; 0.001 (Benjamini–Hochberg FDR)</sup>'
        ),
        x=0.5,
        xanchor='center',
    ),
    width=650,
    height=570,
    margin=dict(l=15, r=70, t=75, b=15),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=13),
)

fig.update_xaxes(
    title_text='',
    showticklabels=True,
    showgrid=False,
    zeroline=False,
)
fig.update_yaxes(
    title_text='',
    showticklabels=True,
    autorange='reversed',
    showgrid=False,
    zeroline=False,
)

save_figure(fig, 'fig03_spearman_heatmap')
fig.show(config={'responsive': True})

### 2.7. Ranking Candidates

*Top 10 unphenotyped clones for each target and model*


In [19]:
# Top 10 unphenotyped candidates per model for the selected target.
import importlib
import src.capstone_pipeline as capstone_pipeline
importlib.reload(capstone_pipeline)  # pick up newly added helpers without restarting the kernel

top_candidates_by_model = capstone_pipeline.top_candidates_by_model
RANKING_TOP_N = 10
target_label = TARGET_NAMES.get(RANKING_TARGET, RANKING_TARGET)
top_by_model = top_candidates_by_model(PHENOTYPE_PATH, GENOTYPE_PATH, target=RANKING_TARGET, top_n=RANKING_TOP_N)
top_by_model['model'] = top_by_model['model_family'].map(MODEL_NAMES)
top10_table = top_by_model.pivot(index='rank', columns='model', values=['clone_id', 'predicted_value'])
top10_table = top10_table.swaplevel(axis=1).reindex(columns=[(MODEL_NAMES[m], c) for m in MODEL_NAMES if m in set(top_by_model['model_family']) for c in ('clone_id', 'predicted_value')])
top10_table.to_csv(OUTPUT_DIR / f'top10_candidates_by_model_{RANKING_TARGET}.csv')
print(f'Top {RANKING_TOP_N} unphenotyped candidates by predicted {target_label}, per model')
top10_table.style.format({c: '{:.3f}' for c in top10_table.columns if c[1] == 'predicted_value'})


Top 10 unphenotyped candidates by predicted Market Index, per model


*Top candidates selected by at least 2 models*

In [20]:
# Candidates selected by at least two models.
print(f"Candidates selected by at least two models for predicted {target_label}")
shared_candidates = top_by_model.groupby('clone_id').filter(
    lambda group: group['model'].nunique() >= 2
)
shared_candidates[['clone_id', 'model', 'rank']].sort_values(['clone_id', 'rank'])

Candidates selected by at least two models for predicted Market Index


,clone_id,model,rank
21,LSU_04752,MLP,2
3,LSU_04752,Elastic Net,4
23,NCSU_08294,MLP,4
4,NCSU_08294,Elastic Net,5
0,USDA_03599,Elastic Net,1
24,USDA_03599,MLP,5


### 2.8. Overfitting Control

The pipeline fits preprocessing inside each outer training fold.   
Elastic Net and MLP limit marker counts   
MLP adds PCA and a small regularized network.  
Ridge retains all SNPs with L2 shrinkage.

## 3. Conclusion
This notebook presents the sweetpotato genomic prediction workflow, including ETL, construction of five indices, comparison of three models using out-of-fold (OOF) predictions, primary-model diagnostics, timing and reload checks, visualization of phenotype associations, and candidate ranking. Because the analysis uses simulated data, the results do not establish predictive accuracy in real sweetpotato populations.